# Tutorial 07: End-to-End Pipeline

This tutorial runs the whole DRP on a simulated observation: raw detector frames
in, a wavelength-calibrated spectrum out. Every stage is exercised at least
once, using the same calls a script would make.

| Stage | What it does | Entry point |
| --- | --- | --- |
| 1. Preprocessing | bias, dark, pixel flat | `image_preprocessing` |
| 2. Reduction | find fibers, trace, extract | `identify_and_trace_fibers`, `extract_spectra` |
| 3. Calibration | rows to wavelengths, line widths | `solve_wavelength_solution`, `fit_line_spread_function` |

Each stage has its own tutorial for the details; this one shows how they fit
together. Tutorials 02, 03, 04 and 05 cover stages 1, 2, 2 and 3 respectively.

**Before you start:** install the package and Jupyter in an environment that
has `numpy`, `astropy`, `matplotlib` and `scipy`; see `docs/CONTRIBUTING.md`.

## Imports

In [ ]:
import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from amasedrp.simulate import synthetic_fiber_flat, synthetic_arc
from amasedrp.preprocessing import image_preprocessing
from amasedrp.reduction import identify_and_trace_fibers, extract_spectra
from amasedrp.calibration import (
    thar_lines,
    solve_wavelength_solution,
    apply_wavelength_solution,
    fit_line_spread_function,
)

## 1. A simulated observation

Five raw frames, written to a temporary directory: a bias, a dark, a pixel flat,
a fiber flat and a ThAr arc. The detector model is the one that matters for the
calibration equation — a bias level, a dark current proportional to exposure
time, and a 2% gradient in pixel response across the detector.

Writing real FITS files rather than passing arrays around means the tutorial
exercises the file-handling path too, and leaves nothing behind in the
repository.

In [ ]:
work = tempfile.mkdtemp(prefix="amasedrp_e2e_")
print(f"working directory: {work}")

n_rows, n_fibers_per_block = 9600, 5
bias_level = 1200.0        # ADU
dark_rate = 1.0            # ADU per second
flat_exptime = 5.0
arc_exptime = 5.0
flat_scale = 3000.0        # ADU at the peak of one fiber
read_noise = 3.0           # ADU per pixel

# Pixel response: a 2% gradient across the detector.
columns = np.linspace(-1.0, 1.0, 210)[np.newaxis, :]
pixel_response = 1.0 + 0.02 * columns

fiber_flat, true_centers = synthetic_fiber_flat(
    n_rows=n_rows, n_blocks=3, n_fibers_per_block=n_fibers_per_block,
)
lines = thar_lines("blue")
arc_image, true_wavelength = synthetic_arc(
    fiber_flat, lines=lines.known_wls,
    wavelength_zero=4390.0, dispersion=0.08221, line_fwhm=0.49,
)


def write_raw(name, signal_adu, exptime, imagetype, seed):
    """Add the detector effects to a signal and write it as a raw frame."""
    shape = (n_rows, pixel_response.shape[1])
    signal = np.broadcast_to(np.asarray(signal_adu, dtype=float), shape)
    data = bias_level + dark_rate * exptime + signal * pixel_response
    data = data + np.random.default_rng(seed).normal(0.0, read_noise, shape)
    header = fits.Header()
    header["IMAGETYP"] = imagetype
    header["EXPTIME"] = exptime
    header["EGAIN"] = 1.0
    path = os.path.join(work, name)
    fits.PrimaryHDU(data.astype(np.float32), header=header).writeto(path)
    return path


paths = {
    "bias": write_raw("bias.fits", 0.0, 0.0, "bias", seed=1),
    "dark": write_raw("dark.fits", 0.0, 60.0, "dark", seed=2),
    "pixflat": write_raw("pixflat.fits", 1000.0 * np.ones_like(fiber_flat),
                         10.0, "flat", seed=3),
    "flat": write_raw("fiber_flat.fits", flat_scale * fiber_flat,
                      flat_exptime, "flat", seed=4),
    "arc": write_raw("arc.fits", flat_scale * arc_image,
                     arc_exptime, "object", seed=5),
}
for name, path in paths.items():
    print(f"  {name:8s} {os.path.basename(path):16s} {os.path.getsize(path) / 1e6:.1f} MB")

## 2. Stage 1: preprocess

`image_preprocessing` applies one equation to the science frame:

```
dark_current = (dark - bias) / dark_exptime
response     = (pixflat - bias - dark_current * pixflat_exptime) / median(...)
calibrated   = (science - bias - dark_current * science_exptime) / response
```

It writes the calibrated frame to disk and returns the `Image`.

In [ ]:
calibrated_frames = {}
for name in ("flat", "arc"):
    calibrated_frames[name] = image_preprocessing(
        input_path=paths[name],
        bias_path=paths["bias"],
        dark_path=paths["dark"],
        pixflat_path=paths["pixflat"],
        output_path=os.path.join(work, f"calibrated_{name}.fits"),
    )
    frame = calibrated_frames[name]
    print(f"{name:5s}: shape {frame.shape}, "
          f"median {np.median(frame.data):.1f} ADU, "
          f"gain {frame.gain} e-/ADU")

sci = calibrated_frames["arc"].data
print(f"\narc frame read back: {sci.shape}, {sci.dtype}")

One thing to check after calibration: the median level of the arc frame should
sit near zero, not near the bias level. If it sits near the bias level, the bias
was never subtracted.

The warning above about negative values is expected here. The arc has no
continuum, and after the bias is removed about half its pixels land below zero
from noise alone. On a real frame the same warning is worth reading: a large
negative tail means the bias was over-subtracted.

In [ ]:
raw = fits.getdata(paths["arc"])
print(f"raw arc median      : {np.median(raw):8.1f} ADU")
print(f"calibrated median   : {np.median(sci):8.1f} ADU")
print(f"bias frame median   : {np.median(fits.getdata(paths['bias'])):8.1f} ADU")

## 3. Stage 2: identify, trace, extract

The calibrated fiber flat says where the fibers are. The calibrated arc is
extracted through that trace model.

In [ ]:
flat_data = calibrated_frames["flat"].data

fibermap, tracemask = identify_and_trace_fibers(
    image=flat_data,
    n_blocks_expected=3,
    n_fibers_per_block_expected=n_fibers_per_block,
    poly_deg=3,
)
frame = extract_spectra(
    image=sci,
    tracemask=tracemask,
    fibermap=fibermap,
    method="boxcar",
    aperture_radius=3,
)

print(f"fibers found : {fibermap.n_fibers} in "
      f"{len(np.unique(fibermap['BLOCKID']))} blocks")
print(f"frame        : {frame.shape}")

residual = tracemask.eval(np.arange(n_rows)) - np.asarray(true_centers)[:, np.newaxis]
spans = tracemask.domain[:, 1] - tracemask.domain[:, 0]
print(f"trace model  : degree {tracemask.coeffs.shape[1] - 1}, "
      f"shortest domain {int(spans.min())} rows")
print(f"trace error  : rms {np.sqrt(np.mean(residual ** 2)):.4f} px, "
      f"max {np.max(np.abs(residual)):.4f} px")

## 4. Stage 3: wavelength calibration

The arc is what makes this possible. Its lines have known wavelengths, so the
pixel-to-wavelength mapping can be solved from the frame itself.

In [ ]:
solution = solve_wavelength_solution(
    frame,
    tracemask,
    known_wls=lines.known_wls,
    poss_wls=lines.poss_wls,
    min_deg=3,
)
calibrated = apply_wavelength_solution(frame, solution)

print(f"fibers solved  : {solution.calibrated_fraction:.1%}")
print(f"median score   : {np.nanmedian(solution.scores):.5f}")
print(f"wave array     : {calibrated.wave.shape}")

rows = np.arange(calibrated.n_wave)
recovered = solution.eval(rows)[0]
error = recovered - true_wavelength
line_rows = np.round((lines.known_wls - true_wavelength[0]) / 0.08221).astype(int)
low, high = int(line_rows.min()), int(line_rows.max())
inside = (rows >= low) & (rows <= high)
print(f"error in the arc-line range : rms {np.sqrt(np.mean(error[inside] ** 2)):.4f} A")
print(f"error outside it            : rms {np.sqrt(np.mean(error[~inside] ** 2)):.4f} A")

In [ ]:
lsf = fit_line_spread_function(calibrated, lines.lsf_wls)

print(f"line-spread function, fwhm in A:")
for index, target in enumerate(lsf.target_wls):
    width = np.nanmedian(lsf.fwhm[index])
    print(f"  {target:8.3f} A : {width:.4f} A, R = {target / width:.0f}")

## 5. The product

The calibrated frame is what leaves the pipeline. Its x-axis is in Angstrom, it
carries a variance estimate, and the fiber map records the quality of each
fiber's wavelength solution.

In [ ]:
product = os.path.join(work, "calibrated_spectra.fits")
calibrated.to_fits(product, overwrite=True)

print(f"written: {product}")
print(f"  flux   {calibrated.flux.shape}, wave {calibrated.wave.shape}")
print(f"  meta   {calibrated.meta}")
print(f"  fiber columns {calibrated.fibermap.colnames}")
print()
print("files produced by this tutorial:")
for name in sorted(os.listdir(work)):
    size = os.path.getsize(os.path.join(work, name)) / 1e6
    print(f"  {name:26s} {size:6.1f} MB")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))

index = 7
axes[0].plot(calibrated.wave[index], calibrated.flux[index], lw=0.7)
axes[0].set_xlabel("wavelength [A]")
axes[0].set_ylabel("flux [ADU]")
axes[0].set_title(f"Calibrated spectrum, fiber {index}")

for target in lines.lsf_wls:
    column = int(np.argmin(np.abs(calibrated.wave[index] - target)))
    lo, hi = column - 12, column + 13
    axis = calibrated.wave[index, lo:hi]
    flux = calibrated.flux[index, lo:hi]
    axes[1].plot(axis - target, flux / np.nanmax(flux), lw=1.0,
                 label=f"{target:.1f} A")
axes[1].set_xlabel("wavelength offset [A]")
axes[1].set_ylabel("normalised flux")
axes[1].set_title("Arc lines of fiber 7")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Cleanup

The tutorial wrote into a temporary directory. Remove it when you are done
looking at the products:

```python
import shutil
shutil.rmtree(work)
```

## Summary

| Stage | Call | Product |
| --- | --- | --- |
| Preprocessing | `image_preprocessing(...)` | calibrated `Image`, written to FITS |
| Reduction | `identify_and_trace_fibers`, `extract_spectra` | `FiberMap`, `TraceMask`, `FiberFrame` |
| Calibration | `solve_wavelength_solution`, `apply_wavelength_solution` | `WavelengthSolution`, per-fiber `wave` |
| Line width | `fit_line_spread_function` | `LineSpreadFunction` |

What to check at each stage, and what this synthetic observation gives:

| Stage | Check | Value here |
| --- | --- | --- |
| Preprocessing | median of a calibrated frame | near 0 ADU, not the 1200 ADU bias |
| Reduction | fibers and blocks found | 15 fibers in 3 blocks |
| Reduction | trace residual against the truth | 0.07 px rms, 0.15 px max |
| Calibration | fibers that solved | 100% |
| Calibration | wavelength error in the arc-line range | 0.08 A rms |
| Calibration | line-spread function | 0.49 A, R 9486 to 10134 |

**The pipeline is only as good as its inputs.** The bias must be subtracted
before extraction, the flat must show the fibers, and the arc must be taken
with the same fiber configuration as the science frame. Every failure mode in
this pipeline traces back to one of those three.

**Where to go next:** `experiments/run_collimator_sweep.py` drives this chain
over a real collimator sweep, and `docs/ARCHITECTURE.md` describes the design
decisions behind it.